# Zero-shot LLM OCR baseline via Groq (Llama vision models)

**Stage:** 1 · Data collection / baseline · **Status:** archived baseline · **Runtime:** local Jupyter, CPU + Groq API

Sanity check before any training: can an off-the-shelf multimodal LLM read Elder Futhark from a photo when given a detailed rune reference table in the prompt? Twelve hand-verified inscriptions are each read three times per model at temperature 0. Metrics: exact match, positional character accuracy and normalized edit distance.

| | |
|---|---|
| **Inputs** | 12 photos named `<id>.png/.jpg` (`GROQ_IMAGE_DIR`) |
| **Outputs** | `validation_<ts>.json`, `summary_<ts>.txt` |

> Set `GROQ_API_KEY` in the environment before running.

## 1. Elder Futhark reference (24 runes ↔ Latin)

In [ ]:
letters = """ᚠ	ᚢ	ᚦ	ᚨ	ᚱ	ᚲ	ᚷ	ᚹ	ᚺ	ᚾ	ᛁ	ᛃ	ᛈ	ᛇ	ᛉ	ᛊ	ᛏ	ᛒ	ᛖ	ᛗ	ᛚ	ᛜ	ᛞ	ᛟ
f	u	þ	a	r	k	g	w	h	n	i	j	p	ï	z	s	t	b	e	m	l	ŋ	d	o"""
letters = letters.split()

In [12]:
print(letters)

['ᚠ', 'ᚢ', 'ᚦ', 'ᚨ', 'ᚱ', 'ᚲ', 'ᚷ', 'ᚹ', 'ᚺ', 'ᚾ', 'ᛁ', 'ᛃ', 'ᛈ', 'ᛇ', 'ᛉ', 'ᛊ', 'ᛏ', 'ᛒ', 'ᛖ', 'ᛗ', 'ᛚ', 'ᛜ', 'ᛞ', 'ᛟ', 'f', 'u', 'þ', 'a', 'r', 'k', 'g', 'w', 'h', 'n', 'i', 'j', 'p', 'ï', 'z', 's', 't', 'b', 'e', 'm', 'l', 'ŋ', 'd', 'o']


## 2. Prompt
The prompt is kept exactly as it was used in the run.

In [ ]:
PROMPT_TEXT = """You are an expert epigrapher specializing in Elder Futhark runic inscriptions (200-700 CE).

## YOUR TASK
Perform accurate OCR of the runic carving shown in the image.

## ELDER FUTHARK REFERENCE TABLE
Use ONLY these 24 runes:
ᚠ = F  (two diagonal branches pointing right from vertical)
ᚢ = U  (V-shape or two diagonals going down)
ᚦ = TH (vertical with triangle/thorn on right)
ᚨ = A  (vertical with two diagonals forming peak at top)
ᚱ = R  (vertical with diagonal leg going right-down)
ᚲ = K  (vertical with two diagonals pointing left)
ᚷ = G  (vertical with X or diagonal cross)
ᚹ = W  (vertical with bent branches)
ᚺ = H  (two verticals connected by horizontal)
ᚾ = N  (two verticals with diagonal between them)
ᛁ = I  (single vertical stroke)
ᛃ = J  (vertical with curved/diagonal branch)
ᛈ = P  (vertical with triangle pointing left)
ᛇ = EI (vertical with multiple horizontals)
ᛉ = Z  (vertical with X branches at top and bottom)
ᛊ = S  (diagonal zigzag or lightning bolt)
ᛏ = T  (vertical with two short diagonals at top, arrow-like)
ᛒ = B  (vertical with two bumps/triangles on right)
ᛖ = E  (vertical with three horizontals)
ᛗ = M  (two verticals connected with arch)
ᛚ = L  (vertical with single diagonal going right-down)
ᛜ = NG (diamond or rhombus shape)
ᛞ = D  (vertical with diamond/lozenge)
ᛟ = O  (diamond with extending verticals)

## READING GUIDELINES
- Read strictly left to right
- Stone carvings may be weathered — match to closest rune shape
- Watch for bind runes (overlapping runes sharing a stave)
- Vertical main stroke (stave) is common to most runes — identify branches carefully
- Ignore decorative scratches or damage marks

## OUTPUT FORMAT
Return **omly** the rune characters in sequence.
No spaces. No translation. No explanation.
Example output: ᚠᚢᚦᚨᚱᚲ"""

## 3. Golden subset and run configuration

In [ ]:
import base64
import os

from groq import Groq

client = Groq(api_key=os.environ["GROQ_API_KEY"])  # set in the environment

# ==================== GOLDEN DATASET ====================
GOLDEN_DATASET = [
    {"id": "6_1", "latin": "bidawarijaztalgidai", "runes": "ᛒᛁᛞᚨᚹᚨᚱᛁᛃᚨᛉᛏᚨᛚᚷᛁᛞᚨᛁ"},
    {"id": "13_0", "latin": "lua", "runes": "ᛚᚢᚨ"},
    {"id": "14_1", "latin": "owlþuþewaz", "runes": "ᛟᚹᛚᚦᚢᚦᛖᚹᚨᛉ"},
    {"id": "15_1", "latin": "ansgzh", "runes": "ᚨᚾᛊᚷᛉᚺ"},
    {"id": "16_3_2", "latin": "ala", "runes": "ᚨᛚᚨ"},
    {"id": "39_1", "latin": "leþro", "runes": "ᛚᛖᚦᚱᛟ"},
    {"id": "42_2", "latin": "aluko", "runes": "ᚨᛚᚢᚲᛟ"},
    {"id": "50_1", "latin": "heþrodwe^ngk", "runes": "ᚺᛖᚦᚱᛟᛞᚹᛖᚷᚲ"},
    {"id": "53_1", "latin": "ekgudijaungandizih", "runes": "ᛖᚲᚷᚢᛞᛁᛃᚨᚢᛜᚨᚾᛞᛁᛉᛁᚺ"},
    {"id": "82_1", "latin": "saligastiz", "runes": "ᛊᚨᛚᛁᚷᚨᛊᛏᛁᛉ"},
    {"id": "263_14", "latin": "oþaainri farkanio", "runes": "ᛟᚦᚨᚨᛁᚾᚱᛁ ᚠᚨᚱᚲᚨᚾᛟ"},
    {"id": "7404", "latin": "lit at fa", "runes": "ᛚᛁᛏ ᚨᛏ ᚠᚨ"},
]

IMAGE_DIR = os.environ.get("GROQ_IMAGE_DIR", "VAL_DATASET_GROQ")  # folder with <id>.png/.jpg photos
EXTENSIONS = (".png", ".jpg", ".jpeg", ".webp", ".gif")
NUM_RUNS = 3

## 4. Models and metrics

In [ ]:
MODELS = [
    "meta-llama/llama-4-scout-17b-16e-instruct",
    "llama-3.2-11b-vision-preview",
    "llama-3.2-90b-vision-preview",
]

# ==================== METRICS ====================


def character_accuracy(pred: str, gold: str) -> float:
    """Per-character accuracy (how many chars match positionally)"""
    if not gold:
        return 1.0 if not pred else 0.0
    matches = sum(p == g for p, g in zip(pred, gold))
    return matches / max(len(pred), len(gold))


def exact_match(pred: str, gold: str) -> bool:
    return pred.strip() == gold.strip()


def levenshtein(s1: str, s2: str) -> int:
    """Edit distance between two strings"""
    m, n = len(s1), len(s2)
    dp = list(range(n + 1))
    for i in range(1, m + 1):
        prev = dp[:]
        dp[0] = i
        for j in range(1, n + 1):
            if s1[i - 1] == s2[j - 1]:
                dp[j] = prev[j - 1]
            else:
                dp[j] = 1 + min(prev[j], dp[j - 1], prev[j - 1])
    return dp[n]


def normalized_edit_distance(pred: str, gold: str) -> float:
    """0.0 = perfect, 1.0 = completely wrong"""
    if not gold and not pred:
        return 0.0
    dist = levenshtein(pred, gold)
    return dist / max(len(pred), len(gold), 1)


# ==================== IMAGE UTILS ====================


def find_image(image_id: str) -> str | None:
    """Find image file by ID (filename without extension)"""
    for fname in os.listdir(IMAGE_DIR):
        name, ext = os.path.splitext(fname)
        if name == image_id and ext.lower() in EXTENSIONS:
            return os.path.join(IMAGE_DIR, fname)
    return None


def encode_image(path: str) -> tuple[str, str]:
    ext = os.path.splitext(path)[1].lower()
    mime = "image/png" if ext == ".png" else "image/jpeg"
    with open(path, "rb") as f:
        return base64.b64encode(f.read()).decode("utf-8"), mime

## 5. Validation loop

In [40]:
import json
import time
from datetime import datetime


def run_groq(image_path: str, model: str) -> str:
    image_data, mime = encode_image(image_path)
    response = client.chat.completions.create(
        model=model,
        messages=[
            {
                "role": "user",
                "content": [
                    {"type": "image_url", "image_url": {"url": f"data:{mime};base64,{image_data}"}},
                    {"type": "text", "text": PROMPT_TEXT},
                ],
            }
        ],
        max_tokens=200,
        temperature=0.0,
    )
    return response.choices[0].message.content.strip()


# ==================== VALIDATION LOOP ====================


def run_validation():
    timestamp = datetime.now().strftime("%Y%m%d_%H%M%S")
    log_path = os.path.join(IMAGE_DIR, f"validation_{timestamp}.json")
    summary_path = os.path.join(IMAGE_DIR, f"summary_{timestamp}.txt")

    all_results = {}  # model -> list of per-sample results

    for model in MODELS:
        print(f"\n{'=' * 60}")
        print(f"MODEL: {model}")
        print(f"{'=' * 60}")
        all_results[model] = []

        for sample in GOLDEN_DATASET:
            sid = sample["id"]
            gold = sample["runes"]

            image_path = find_image(sid)
            if not image_path:
                print(f"  [{sid}] ⚠️  Изображение не найдено, пропускаю")
                continue

            sample_result = {"id": sid, "gold": gold, "runs": []}

            for run_idx in range(1, NUM_RUNS + 1):
                print(f"  [{sid}] run {run_idx}/{NUM_RUNS} ...", end=" ", flush=True)
                try:
                    pred = run_groq(image_path, model)
                    em = exact_match(pred, gold)
                    cha = character_accuracy(pred, gold)
                    ned = normalized_edit_distance(pred, gold)

                    run_data = {
                        "run": run_idx,
                        "prediction": pred,
                        "exact_match": em,
                        "char_accuracy": round(cha, 4),
                        "norm_edit_dist": round(ned, 4),
                    }
                    sample_result["runs"].append(run_data)

                    status = "✅" if em else "❌"
                    print(f"{status}  pred='{pred}'  gold='{gold}'  CHA={cha:.2f}  NED={ned:.2f}")

                except Exception as e:
                    print(f"💥 Ошибка: {e}")
                    sample_result["runs"].append({"run": run_idx, "error": str(e)})

                time.sleep(4)  # rate limit

            all_results[model].append(sample_result)

    # ==================== AGGREGATE METRICS ====================

    print(f"\n{'=' * 60}")
    print("ИТОГОВЫЕ МЕТРИКИ")
    print(f"{'=' * 60}")

    summary_lines = []
    json_output = {"timestamp": timestamp, "models": {}}

    for model, samples in all_results.items():
        em_scores, cha_scores, ned_scores = [], [], []

        for sample in samples:
            for run in sample["runs"]:
                if "error" in run:
                    continue
                em_scores.append(int(run["exact_match"]))
                cha_scores.append(run["char_accuracy"])
                ned_scores.append(run["norm_edit_dist"])

        if not em_scores:
            continue

        avg_em = sum(em_scores) / len(em_scores)
        avg_cha = sum(cha_scores) / len(cha_scores)
        avg_ned = sum(ned_scores) / len(ned_scores)

        line = (
            f"\n{model}\n"
            f"  Exact Match:          {avg_em * 100:.1f}%  ({sum(em_scores)}/{len(em_scores)})\n"
            f"  Char Accuracy:        {avg_cha * 100:.1f}%\n"
            f"  Norm Edit Distance:   {avg_ned:.4f}  (0=perfect)\n"
        )
        print(line)
        summary_lines.append(line)

        json_output["models"][model] = {
            "exact_match_pct": round(avg_em * 100, 2),
            "char_accuracy_pct": round(avg_cha * 100, 2),
            "norm_edit_dist": round(avg_ned, 4),
            "total_runs": len(em_scores),
            "samples": samples,
        }

    # ==================== SAVE LOGS ====================

    with open(log_path, "w", encoding="utf-8") as f:
        json.dump(json_output, f, ensure_ascii=False, indent=2)

    with open(summary_path, "w", encoding="utf-8") as f:
        f.write(f"Validation run: {timestamp}\n")
        f.writelines(summary_lines)

    print(f"\n📄 JSON лог:    {log_path}")
    print(f"📄 Summary:     {summary_path}")


run_validation()


MODEL: meta-llama/llama-4-scout-17b-16e-instruct
  [6_1] run 1/3 ... ❌  pred='## Step 1: Examine the image and identify the runic carvings
The image shows a broken stone or ceramic piece with runic inscriptions. The visible runes appear to be in the Elder Futhark.

## 2: Transcribe the runes from left to right
Starting from the left, the first rune appears to be ᛒ, followed by ᛁ, then ᛃ, then ᛗ, then ᚴ, then ᛋ, then ᛏ, then ᚾ, then ᛁ, then ᚷ, then ᚾ, then ᚹ.

## 3: Verify the runes against the Elder Futhark reference table
Verifying each rune against the provided reference table:
- ᛒ = B
- ᛁ = I
- ᛃ = J
- ᛗ = M
- ᚴ = K
- ᛋ = S'  gold='ᛒᛁᛞᚨᚹᚨᚱᛁᛃᚨᛉᛏᚨᛚᚷᛁᛞᚨᛁ'  CHA=0.00  NED=0.99
  [6_1] run 2/3 ... ❌  pred='## Step 1: Examine the Image
The image shows a runic inscription on a broken stone or ceramic piece. The inscription appears to be in Elder Futhark runes.

## 2: Identify Individual Runes
Starting from the left, the first rune appears to be ᛒ. The second rune looks like ᛁ. The third ru

## Result
* **Llama-4-Scout-17B** ignored the output format: every answer began with a chain of reasoning
  (`## Step 1: Examine the image…`) instead of runes, so exact match was **0 %** on all runs.
* **Llama-3.2-11B/90B-Vision** had already been decommissioned by Groq (HTTP 400), and the run was stopped manually.

Prompting a general-purpose LLM does not give usable runic OCR. This motivated the
synthetic-data + fine-tuning approach of the thesis.